# 03 (v1.1 연습). 검색 베이스라인 + 재정렬

`T2_v11_embedding (test).ipynb`가 만든 **v1.1** 의료 DB(문단 142,605개)를 그대로 쓰고, **LLM 없이** 검색 쪽 베이스라인과 재정렬까지만 만듭니다. claim 생성(HyDE)과 답변 생성은 다음 단계입니다.

**비교하는 방법**

| ID | 문서를 뽑는 방법 | 실제 사용 가능? |
|---|---|---|
| Q-dense | 질문 → bge 임베딩 검색 (02와 같음) | ✅ |
| Q-bm25 | 질문 → BM25 단어 검색 | ✅ |
| Q-hybrid | 위 두 결과를 순위로 합침 (RRF) | ✅ |
| C-oracle | 질문 + **정답 문장**으로 만든 claim → 검색 | ❌ 정답을 쓰므로 치팅. **참고용 상한선**만 |

그다음 1단계 후보 100개를 **cross-encoder 재정렬**로 다시 줄 세웁니다.

**데이터 나누기 (치팅 방지)**
- **검증(val) 200개**: 평가 질문과 겹치지 않는 dev 의료 질문. 방법·설정은 여기 점수로만 고릅니다.
- **시험(test) 500개**: 기존 `eval.jsonl`의 답 있는 질문. 마지막에 한 번 채점합니다.

결과는 드라이브 `rag_msmarco/v11_test/rerank/`에 저장되고, 생성 담당(B)에게 넘길 **질문별 상위 20개 문단 번호** 파일도 만듭니다.

**런타임: A100.** 위에서부터 순서대로 실행하세요. 마지막 서버 종료 셀은 직접 켜야 동작합니다.

## 1. 설치 + 저장 위치

In [ ]:
!pip -q install sentence-transformers faiss-cpu bm25s

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/rag_msmarco/v11_test'    # T2 (v1.1) 노트북 결과 폴더
except Exception as e:
    raise RuntimeError('드라이브 연결이 필요합니다. T2 (v1.1) 노트북 결과가 드라이브에 있어요.') from e
OUT_DIR = f'{SAVE_DIR}/rerank'
print('DB 위치:', SAVE_DIR, '/ 결과 저장:', OUT_DIR)

## 2. 설정

`LIMIT`에 숫자를 넣으면 질문을 그만큼만 써서 빠르게 시험합니다. 본 실행은 `None`.

In [ ]:
import os, re, json, random, time
from collections import Counter
import numpy as np
import torch

SEED         = 42
N_VAL        = 200        # 검증 질문 수
K            = 100        # 1단계에서 뽑을 후보 수 (= 재정렬할 후보 수)
RRF_K        = 60         # RRF 결합 상수 (관례값)
LIMIT        = None       # 빠른 시험용. 예: 20
MIN_HEALTH_HITS = 2       # 02와 같은 의료 판정 기준

EMB_MODEL    = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKERS    = ['cross-encoder/ms-marco-MiniLM-L-6-v2', 'BAAI/bge-reranker-base']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(OUT_DIR, exist_ok=True)
print('장치:', DEVICE)

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

## 3. DB 불러오기 (02 결과)

In [ ]:
import faiss

corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
print(f'문단 {len(corpus):,} / 인덱스 {index.ntotal:,} / 평가 질문 {len(eval_set)}')
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'

## 4. 검증 세트 만들기 (val 200)

dev 의료 질문 중 **평가 질문과 겹치지 않고**, 정답 문단이 DB에 있는 질문 200개를 고릅니다.
02와 같은 정리 함수로 문단 내용을 맞춰서 정답 문단 번호를 찾습니다.

In [ ]:
from datasets import load_dataset
from urllib.parse import urlparse

HEALTH_SITES = [
    'webmd.com', 'mayoclinic.org', 'medicinenet.com', 'medical-dictionary.thefreedictionary.com',
    'verywell.com', 'healthline.com', 'drugs.com', 'livestrong.com', 'healthtap.com', 'cdc.gov',
    'rightdiagnosis.com', 'emedicinehealth.com', 'medscape.com', 'everydayhealth.com',
    'medlineplus.gov', 'nhs.uk', 'rxlist.com', 'sharecare.com', 'medicalnewstoday.com',
    'kidshealth.org', 'nih.gov', 'patient.info', 'labtestsonline.org', 'aafp.org',
    'clevelandclinic.org', 'hopkinsmedicine.org', 'healthgrades.com', 'familydoctor.org',
    'merckmanuals.com', 'uptodate.com', 'health.harvard.edu', 'babycenter.com', 'psychologytoday.com',
]
def is_health(url):
    h = urlparse(url).netloc.lower()
    h = h[4:] if h.startswith('www.') else h
    return any(h == s or h.endswith('.' + s) for s in HEALTH_SITES)

def clean_query(q):
    q = ' '.join(q.split())
    return re.sub(r'^[\s.,;:!?\-]+', '', q)

def clean_text(t):
    return ' '.join(t.split())

NO_ANSWER = 'No Answer Present.'
pid_of   = {t: i for i, t in enumerate(texts)}
eval_qid = {q['qid'] for q in eval_set}

dev = load_dataset('microsoft/ms_marco', 'v1.1')['validation']
cands = []
for b in dev.iter(batch_size=5000):
    for j in range(len(b['query'])):
        p, qid = b['passages'][j], int(b['query_id'][j])
        if qid in eval_qid or sum(is_health(u) for u in p['url']) < MIN_HEALTH_HITS:
            continue
        answers = [clean_text(a) for a in b['answers'][j] if a and a.strip()]
        if not answers or answers == [NO_ANSWER]:
            continue
        gold = [pid_of[clean_text(t)] for t, s in zip(p['passage_text'], p['is_selected'])
                if s == 1 and clean_text(t) in pid_of]
        if gold:
            cands.append({'qid': qid, 'query': clean_query(b['query'][j]), 'gold_pids': sorted(set(gold)),
                          'answers': answers, 'answerable': True,
                          'query_type': b['query_type'][j], 'domain': 'health'})

val_set = random.Random(SEED).sample(cands, min(N_VAL, len(cands)))
test_set = [q for q in eval_set if q['answerable']]
assert not {q['qid'] for q in val_set} & {q['qid'] for q in eval_set}, '검증과 평가 질문이 겹침'
if LIMIT:
    val_set, test_set = val_set[:LIMIT], test_set[:LIMIT]
save_jsonl(val_set, f'{OUT_DIR}/val.jsonl')
print(f'검증 후보 {len(cands):,}개 중 {len(val_set)}개 선택 / 시험 {len(test_set)}개 (겹침 없음 확인)')

## 5. 공통 함수 + 모델 준비

- **Recall@k**: 상위 k개 안에 정답 문단이 하나라도 있는 질문 비율
- **MRR@10**: 첫 정답 순위의 역수 평균 (MS MARCO 공식 지표)
- **RRF**: 여러 검색 결과를 순위만으로 합치는 방법. 점수 = Σ 1 / (60 + 순위)

In [ ]:
from sentence_transformers import SentenceTransformer
import bm25s

def metrics(ranked, qs, ks=(1, 5, 10, 100)):
    res = {f'R@{k}': 0.0 for k in ks}
    res['MRR@10'] = 0.0
    for pids, q in zip(ranked, qs):
        gold = set(q['gold_pids'])
        for k in ks:
            res[f'R@{k}'] += bool(gold & set(pids[:k]))
        for r, p in enumerate(pids[:10]):
            if p in gold:
                res['MRR@10'] += 1 / (r + 1)
                break
    return {k: round(v / len(qs), 4) for k, v in res.items()}

def rrf(*rankings, k=RRF_K, top=K):
    out = []
    for lists in zip(*rankings):
        score = Counter()
        for lst in lists:
            for r, p in enumerate(lst):
                score[p] += 1 / (k + r + 1)
        out.append([p for p, _ in score.most_common(top)])
    return out

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
if DEVICE == 'cuda':
    emb_model.half()

def encode(strings, prefix=''):
    return emb_model.encode([prefix + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

def dense_search(vecs, top=K):
    _, ids = index.search(vecs, top)
    return ids.tolist()

t = time.time()
bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize(texts, stopwords='en', show_progress=False), show_progress=False)
print(f'BM25 인덱스 {time.time() - t:.0f}초')

def bm25_search(strings, top=K):
    ids, _ = bm25.retrieve(bm25s.tokenize(strings, stopwords='en', show_progress=False), k=top, show_progress=False)
    return ids.tolist()

splits = {'val': val_set, 'test': test_set}
runs = {s: {} for s in splits}          # runs[split][방법] = 질문별 후보 pid 목록

## 6. 질문으로 뽑기 (Q-dense, Q-bm25, Q-hybrid)

In [ ]:
q_vecs = {}
for s, qs in splits.items():
    queries = [q['query'] for q in qs]
    q_vecs[s] = encode(queries, QUERY_PREFIX)
    runs[s]['Q-dense']  = dense_search(q_vecs[s])
    runs[s]['Q-bm25']   = bm25_search(queries)
    runs[s]['Q-hybrid'] = rrf(runs[s]['Q-dense'], runs[s]['Q-bm25'])
    for m in ['Q-dense', 'Q-bm25', 'Q-hybrid']:
        print(f'[{s}] {m:9s}', metrics(runs[s][m], qs))

## 7. 참고: 정답 claim (C-oracle, 치팅 상한선)

질문 + **정답 문장**으로 claim을 만들어 검색합니다. 실제 서비스에서는 정답을 모르니 **쓸 수 없는 방법**입니다.
"claim이 완벽하면 검색이 어디까지 되나"를 보는 기준으로만 쓰고, 최종 선택 후보에서는 뺍니다.

In [ ]:
for s, qs in splits.items():
    oracle = [f"{q['query']} {q['answers'][0]}" for q in qs]
    runs[s]['C-oracle'] = dense_search(encode(oracle))
    print(f'[{s}] C-oracle  ', metrics(runs[s]['C-oracle'], qs), '← 참고용 (치팅)')

## 8. 재정렬 (cross-encoder)

임베딩 검색은 질문과 문단을 **따로** 벡터로 만들어 비교합니다. cross-encoder는 질문과 문단을 **한 번에 같이 읽고** 관련도를 매겨서 더 정확하지만 느립니다.
그래서 1단계에서 뽑은 후보 100개만 다시 줄 세웁니다.

1단계 방법은 **검증 세트의 R@100이 가장 높은 것**을 자동으로 고릅니다 (C-oracle 제외). 비교를 위해 Q-dense 후보도 함께 재정렬합니다.

In [ ]:
from sentence_transformers import CrossEncoder

usable = [m for m in runs['val'] if m != 'C-oracle']
best_first = max(usable, key=lambda m: metrics(runs['val'][m], splits['val'])['R@100'])
first_stages = list(dict.fromkeys([best_first, 'Q-dense']))
print('1단계 선택 (검증 R@100 기준):', best_first)

CE_BATCH = 256

def check_reranker(ce):
    """관련 있는 문장이 더 높은 점수를 받는지, 점수가 nan이 아닌지 확인합니다."""
    s = ce.predict([('how many people live in berlin', 'Berlin has a population of about 3.5 million people.'),
                    ('how many people live in berlin', 'The cat sat on the mat.')], batch_size=CE_BATCH)
    assert np.isfinite(s).all(), f'재정렬 점수가 nan입니다: {s}. 라이브러리 버전 문제일 수 있어요.'
    assert s[0] > s[1], f'관련 문장 점수가 더 낮습니다: {s}'
    print('  점검 통과:', np.round(s, 2))

def rerank(ce, qs, cand_lists):
    out = []
    for q, cands in zip(qs, cand_lists):
        scores = ce.predict([(q['query'], texts[p]) for p in cands], batch_size=CE_BATCH, show_progress_bar=False)
        assert np.isfinite(scores).all(), '재정렬 중 nan 점수 발생'
        out.append([cands[i] for i in np.argsort(-scores)])
    return out

for name in RERANKERS:
    t = time.time()
    ce = CrossEncoder(name, max_length=256, device=DEVICE)
    check_reranker(ce)
    short = name.split('/')[-1]
    for fs in first_stages:
        for s, qs in splits.items():
            runs[s][f'{fs} → {short}'] = rerank(ce, qs, runs[s][fs])
    print(f'{short} 재정렬 {time.time() - t:.0f}초')
    del ce

## 9. 결과 표

**방법은 검증(val) 점수로 고르고**, 시험(test) 점수는 최종 보고용입니다.

In [ ]:
import pandas as pd

tables = {s: pd.DataFrame({m: metrics(r, splits[s]) for m, r in runs[s].items()}).T for s in splits}
for s in splits:
    print(f'\n[{s}] 질문 {len(splits[s])}개')
    display(tables[s])

final = max([m for m in runs['val'] if m != 'C-oracle'], key=lambda m: metrics(runs['val'][m], splits['val'])['MRR@10'])
print('\n검증 MRR@10 기준 최종 선택:', final)
print('시험 점수:', metrics(runs['test'][final], splits['test']))

## 10. 저장 + 생성 담당에게 넘길 파일

- `results.json`, `results_summary.md`: 점수 표
- `test_top20.jsonl`, `val_top20.jsonl`: 최종 방법으로 뽑은 **질문별 상위 20개 문단 번호**. 다음 생성(LLM) 단계에서 그대로 씁니다.
  (평가 질문 중 답 없는 질문·분야 밖 질문은 다음 셀에서 따로 처리합니다)

In [ ]:
def to_md(df):
    try:
        return df.to_markdown()
    except ImportError:
        return '```\n' + df.to_string() + '\n```'

with open(f'{OUT_DIR}/results.json', 'w') as f:
    json.dump({'final_method': final, 'first_stage_by_val': best_first,
               'n_val': len(splits['val']), 'n_test': len(splits['test']),
               'val': tables['val'].to_dict('index'), 'test': tables['test'].to_dict('index')}, f, indent=2)

summary = '\n'.join([
    f"# 03 문서 뽑기 + 재정렬 결과 ({time.strftime('%Y-%m-%d %H:%M')})", '',
    f'- DB: 문단 {len(corpus):,}개 / 검증 {len(splits["val"])}개 / 시험 {len(splits["test"])}개',
    f'- 1단계 선택(검증 R@100): {best_first} / 최종 선택(검증 MRR@10): {final}',
    '- C-oracle은 정답을 쓰는 참고용 상한선(치팅)', '',
    '## 검증', '', to_md(tables['val']), '', '## 시험', '', to_md(tables['test']),
])
with open(f'{OUT_DIR}/results_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)

for s, qs in splits.items():
    save_jsonl([{'qid': q['qid'], 'query': q['query'], 'top20': [int(p) for p in r[:20]]}
                for q, r in zip(qs, runs[s][final])], f'{OUT_DIR}/{s}_top20.jsonl')
print(summary)

### 답 없는 질문 + 분야 밖 질문도 문단 뽑아 두기

생성 단계에서 "모르면 거절"을 시험하려면 이 질문들에도 검색 결과가 필요합니다. 최종 방법과 같은 방식으로 뽑고 재정렬합니다.

In [ ]:
others = [q for q in eval_set if not q['answerable']]
if LIMIT:
    others = others[:LIMIT]
fs_other = 'Q-hybrid' if 'Q-hybrid' in final else 'Q-dense'
base = {'Q-dense': dense_search(encode([q['query'] for q in others], QUERY_PREFIX))}
base['Q-hybrid'] = rrf(base['Q-dense'], bm25_search([q['query'] for q in others]))
cand = base[fs_other]
if '→' in final:
    ce = CrossEncoder(next(n for n in RERANKERS if n.split('/')[-1] in final), max_length=256, device=DEVICE)
    cand = rerank(ce, others, cand)
save_jsonl([{'qid': q['qid'], 'query': q['query'], 'domain': q['domain'], 'top20': [int(p) for p in r[:20]]}
            for q, r in zip(others, cand)], f'{OUT_DIR}/test_unanswerable_top20.jsonl')
print('답 없음·분야 밖', len(others), '개 저장 →', Counter(q['domain'] for q in others))

## 11. 예시: 재정렬 전후 비교 (✅ = 정답 문단)

In [ ]:
q = splits['test'][0]
print('질문:', q['query'], '\n정답:', q['answers'][0], '\n')
for m in ['Q-dense', final]:
    print(f'[{m}]')
    for p in runs['test'][m][0][:3] if m != final else runs['test'][final][0][:3]:
        mark = '✅' if p in q['gold_pids'] else '  '
        print(f' {mark} [{p}] {corpus[p]["url"]}\n      {texts[p][:150]}')

## 12. 드라이브 저장 마무리 + 런타임 종료

결과 파일은 앞 셀에서 드라이브(`{SAVE_DIR}/rerank/`)에 저장됩니다. 이 셀은 저장을 마무리하고 Colab 런타임을 끕니다.

In [ ]:
Shutdown = True

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()   # 결과 파일 드라이브 저장 마무리
runtime.unassign()          # 런타임 종료
